# Experiment 3 — Document-Grounded Dialogue Generation in Hinglish

This experiment extends the **Experiment 2 LSTM encoder–decoder + Bahdanau attention** architecture to multi-turn, document-grounded Hinglish dialogue.

### Architecture

\[
H = Encoder_{dialogue}(x_h), \qquad D = Encoder_{document}(x_d)
\]

The decoder computes two Bahdanau attention distributions:

\[
c_t^h=\sum_i\alpha^h_{t,i}H_i,\qquad
c_t^d=\sum_j\alpha^d_{t,j}D_j
\]

and combines them with a learned source gate:

\[
g_t=\sigma(W_g[s_{t-1};c_t^h;c_t^d])
\]

\[
c_t=g_t\odot c_t^d+(1-g_t)\odot c_t^h
\]

The notebook contains:

- Hugging Face CMU Hinglish DoG import
- official CMU DoG WikiData join
- conversation reconstruction
- custom Hinglish preprocessing
- learned embeddings from scratch
- ungrounded neural baseline
- TF-IDF retrieval baseline
- grounded dual-source generator
- StratifiedGroupKFold
- OOF predictions
- optional embedding/hidden-size grid search
- validation-based checkpointing
- greedy and beam decoding
- BLEU + ROUGE
- code-mixing degradation analysis
- grounding diagnostics
- qualitative comparison
- artifact saving

## Dataset note

The Hugging Face rows contain `translation.hi_en`, `translation.en`, `wikiDocumentIdx`, and `docIdx`, but the actual WikiData text is stored separately in the official CMU DoG repository.

The notebook therefore loads conversation turns from:

`festvox/cmu_hinglish_dog`

and maps each row to the corresponding official WikiData document using `wikiDocumentIdx`. This prevents a fake "document-grounded" experiment in which the document input is empty.

In [ ]:
# ============================================================
# 1. IMPORTS + REPRODUCIBILITY
# ============================================================

import os, re, json, random, shutil, subprocess
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold, ParameterGrid
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from nltk.translate.bleu_score import corpus_bleu

try:
    from rouge_score import rouge_scorer
except ImportError:
    rouge_scorer = None

from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HF_DATASET_ID = "festvox/cmu_hinglish_dog"
CMU_DOG_REPO = "https://github.com/festvox/datasets-CMU_DoG.git"

RUN_CV = True
RUN_GRID_SEARCH = True

N_SPLITS = 3
CV_EPOCHS = 4
FINAL_EPOCHS = 12
BATCH_SIZE = 64

MAX_HISTORY_TOKENS = 120
MAX_DOCUMENT_TOKENS = 256
MAX_TARGET_TOKENS = 40

MIN_FREQ = 2
DEFAULT_EMB_DIM = 256
DEFAULT_HID_DIM = 384
DROPOUT = 0.25
LR = 1e-3
CLIP = 1.0
TEACHER_FORCING = 0.5
PATIENCE = 2

BEAM_WIDTH = 5
LENGTH_PENALTY = 0.7

ARTIFACT_DIR = Path("/mnt/data/exp3_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)

In [ ]:
# ============================================================
# 2. LOAD HUGGING FACE DATASET
# ============================================================

raw = load_dataset(HF_DATASET_ID)

def tr_value(obj, key):
    value = obj.get(key, "") if isinstance(obj, dict) else ""
    return "" if value is None else str(value)

def to_df(ds, split):
    rows = []
    for i, row in enumerate(ds):
        tr = row.get("translation", {})
        rows.append({
            "row_id": f"{split}_{i}",
            "date": str(row.get("date", "")),
            "utcTimestamp": str(row.get("utcTimestamp", "")),
            "uid": str(row.get("uid", "")),
            "user2_id": str(row.get("user2_id", "")),
            "wikiDocumentIdx": int(row.get("wikiDocumentIdx", -1)),
            "docIdx": int(row.get("docIdx", -1)),
            "hi_en": tr_value(tr, "hi_en"),
            "en": tr_value(tr, "en"),
            "rating": row.get("rating", np.nan),
            "status": row.get("status", np.nan)
        })
    return pd.DataFrame(rows)

train_df = to_df(raw["train"], "train")
val_df = to_df(raw["validation"], "val")
test_df = to_df(raw["test"], "test")

print(train_df.shape, val_df.shape, test_df.shape)
display(train_df.head(3))

In [ ]:
# ============================================================
# 3. CLEANING + CONVERSATION GROUPS
# ============================================================

def clean_text(x):
    x = "" if x is None else str(x)
    x = x.replace("\r", " ").replace("\n", " ").replace("\u00a0", " ")
    return re.sub(r"\s+", " ", x).strip()

def clean_df(df):
    df = df.copy()
    df["hi_en"] = df["hi_en"].map(clean_text)
    df["en"] = df["en"].map(clean_text)

    df = df[
        (df["hi_en"].str.len() > 0) &
        (df["wikiDocumentIdx"] >= 0)
    ].copy()

    # Turns from one dialogue share the session date and user2_id.
    df["conversation_id"] = (
        df["date"].astype(str) + "||" +
        df["user2_id"].astype(str)
    )

    df["timestamp"] = pd.to_datetime(
        df["utcTimestamp"], errors="coerce", utc=True
    )

    return df.sort_values(
        ["conversation_id", "timestamp", "row_id"]
    ).reset_index(drop=True)

train_df = clean_df(train_df)
val_df = clean_df(val_df)
test_df = clean_df(test_df)

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(df), "turns |",
          df["conversation_id"].nunique(), "conversations")

In [ ]:
# ============================================================
# 4. DOWNLOAD + PARSE OFFICIAL WIKIDATA
# ============================================================

REPO_DIR = Path("/mnt/data/cmu_dog_repo")
WIKI_DIR = REPO_DIR / "WikiData"

if not (WIKI_DIR.exists() and list(WIKI_DIR.glob("*.json"))):
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    result = subprocess.run(
        ["git", "clone", "--depth", "1", CMU_DOG_REPO, str(REPO_DIR)],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr)

def flatten(v):
    if v is None:
        return ""
    if isinstance(v, str):
        return v
    if isinstance(v, list):
        return " ".join(flatten(x) for x in v)
    if isinstance(v, dict):
        parts = []
        for k, x in v.items():
            if k == "wikiDocumentIdx":
                continue
            tx = flatten(x)
            if tx.strip():
                parts.append(f"{k}: {tx}")
        return " ".join(parts)
    return str(v)

wiki_docs = {}

for path in sorted(WIKI_DIR.glob("*.json")):
    try:
        obj = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue

    doc_id = obj.get("wikiDocumentIdx", obj.get("wikiDocumentId"))

    if doc_id is None:
        m = re.search(r"(\d+)", path.stem)
        if m:
            doc_id = int(m.group(1))

    if doc_id is None:
        continue

    wiki_docs[int(doc_id)] = {
        s: flatten(obj.get(str(s), ""))
        for s in range(4)
    }

print("WikiData documents:", len(wiki_docs))

In [ ]:
# ============================================================
# 5. JOIN DOCUMENTS + RECONSTRUCT HISTORY
# ============================================================

def make_document(doc_id, section_id, mode="full_document"):
    sections = wiki_docs.get(int(doc_id), {})
    if not sections:
        return ""

    if mode == "current_section":
        return sections.get(int(section_id), "")

    return " ".join(
        f"[SECTION {s}] {sections[s]}"
        for s in sorted(sections)
        if sections[s].strip()
    )

DOCUMENT_MODE = "full_document"

for df in [train_df, val_df, test_df]:
    df["document"] = [
        make_document(d, s, DOCUMENT_MODE)
        for d, s in zip(df["wikiDocumentIdx"], df["docIdx"])
    ]

def build_examples(df):
    examples = []

    for conversation_id, group in df.groupby(
        "conversation_id", sort=False
    ):
        group = group.sort_values(["timestamp", "row_id"])
        history = []

        for _, row in group.iterrows():
            history_text = (
                " ".join(
                    f"[{speaker}] {text}"
                    for speaker, text in history[-8:]
                )
                if history else "[START]"
            )

            examples.append({
                "row_id": row["row_id"],
                "conversation_id": conversation_id,
                "history": history_text,
                "document": row["document"],
                "target": row["hi_en"],
                "docIdx": int(row["docIdx"]),
                "wikiDocumentIdx": int(row["wikiDocumentIdx"])
            })

            history.append((row["uid"], row["hi_en"]))

    return pd.DataFrame(examples)

train_ex = build_examples(train_df)
val_ex = build_examples(val_df)
test_ex = build_examples(test_df)

print(train_ex.shape, val_ex.shape, test_ex.shape)
display(train_ex.head(2))

In [ ]:
# ============================================================
# 6. CUSTOM HINGLISH TOKENIZATION + CODE-MIX DIAGNOSTICS
# ============================================================

TOKEN_PATTERN = re.compile(
    r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+(?:\.\d+)?|[^\w\s]"
)

def tokenize(text):
    return TOKEN_PATTERN.findall(clean_text(text).lower())

def detokenize(tokens):
    text = " ".join(tokens)
    text = re.sub(r"\s+([,.!?;:%)\]}])", r"\1", text)
    text = re.sub(r"([(\\[{])\s+", r"\1", text)
    return text.strip()

HINDI_HINTS = set("""
hai hain ho tha thi the ka ke ki ko se mein me par aur ya nahi kya
tum tumne main mai mujhe mera meri mere aap aapka aapki aapke ye yeh
woh wo uska uski uske kyun kyu kyunki kaise kaafi bahut bhi hi ek
batao bata sakte sakta sakti raha rahi rahe kar raha karte karta
hota hoti hoga hogi hone accha acchi acche bura buri bade chhota
lambi kitni kis bare baare
""".split())

ENGLISH_HINTS = set("""
the a an is are was were be been being i you he she it we they this
that those these what why how who where when do does did can could
will would should have has had movie film great good bad about with
from into on in of for and or but not tell say said see know think
like want much more very really interesting
""".split())

def ratio(text, lexicon):
    toks = [t for t in tokenize(text) if t.isalpha()]
    return sum(t in lexicon for t in toks) / max(1, len(toks))

for df in [train_ex, val_ex, test_ex]:
    df["hinglish_ratio"] = df["target"].map(
        lambda x: ratio(x, HINDI_HINTS)
    )
    df["english_ratio"] = df["target"].map(
        lambda x: ratio(x, ENGLISH_HINTS)
    )

for df in [train_ex, val_ex, test_ex]:
    df["mix_bin"] = pd.qcut(
        df["hinglish_ratio"].rank(method="first"),
        q=3,
        labels=False
    ).astype(int)

display(train_ex["mix_bin"].value_counts().sort_index())

In [ ]:
# ============================================================
# 7. TRAIN-ONLY VOCABULARY + DATASET
# ============================================================

PAD, UNK, SOS, EOS = "<pad>", "<unk>", "<sos>", "<eos>"
SPECIALS = [PAD, UNK, SOS, EOS]

class Vocab:
    def __init__(self, texts, min_freq=2):
        counts = Counter()
        for text in texts:
            counts.update(tokenize(text))

        self.itos = SPECIALS.copy()
        for tok, freq in counts.most_common():
            if freq >= min_freq and tok not in SPECIALS:
                self.itos.append(tok)

        self.stoi = {tok: i for i, tok in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, text, max_len):
        tokens = tokenize(text)[:max_len - 2]
        return (
            [self.stoi[SOS]]
            + [self.stoi.get(t, self.stoi[UNK]) for t in tokens]
            + [self.stoi[EOS]]
        )

    def decode(self, ids):
        tokens = []
        for i in ids:
            tok = self.itos[int(i)]
            if tok in (PAD, SOS):
                continue
            if tok == EOS:
                break
            tokens.append(tok)
        return detokenize(tokens)

src_vocab = Vocab(
    train_ex["history"].tolist()
    + train_ex["document"].tolist(),
    MIN_FREQ
)
trg_vocab = Vocab(train_ex["target"].tolist(), MIN_FREQ)

class DialogueDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        return {
            "history": src_vocab.encode(
                row["history"], MAX_HISTORY_TOKENS
            ),
            "document": src_vocab.encode(
                row["document"], MAX_DOCUMENT_TOKENS
            ),
            "target": trg_vocab.encode(
                row["target"], MAX_TARGET_TOKENS
            ),
            "row_id": row["row_id"],
            "conversation_id": row["conversation_id"]
        }

def pad_batch(seqs, pad_idx):
    out = torch.full(
        (len(seqs), max(map(len, seqs))),
        pad_idx,
        dtype=torch.long
    )
    for i, seq in enumerate(seqs):
        out[i, :len(seq)] = torch.tensor(seq)
    return out

def collate(batch):
    return {
        "history": pad_batch(
            [x["history"] for x in batch],
            src_vocab.stoi[PAD]
        ),
        "document": pad_batch(
            [x["document"] for x in batch],
            src_vocab.stoi[PAD]
        ),
        "target": pad_batch(
            [x["target"] for x in batch],
            trg_vocab.stoi[PAD]
        ),
        "row_id": [x["row_id"] for x in batch],
        "conversation_id": [x["conversation_id"] for x in batch]
    }

def loader(df, shuffle=False):
    return DataLoader(
        DialogueDataset(df),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate,
        pin_memory=torch.cuda.is_available()
    )

print("Source vocab:", len(src_vocab))
print("Target vocab:", len(trg_vocab))

In [ ]:
# ============================================================
# 8. EXPERIMENT-2 BAHdanau ATTENTION + TWO ENCODERS
# ============================================================

class BahdanauAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim, attn_dim):
        super().__init__()
        self.enc = nn.Linear(enc_dim, attn_dim, bias=False)
        self.dec = nn.Linear(dec_dim, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, enc_out, hidden, mask):
        energy = self.v(
            torch.tanh(
                self.enc(enc_out)
                + self.dec(hidden).unsqueeze(1)
            )
        ).squeeze(-1)

        energy = energy.masked_fill(~mask, -1e9)
        alpha = torch.softmax(energy, dim=1)
        context = torch.bmm(
            alpha.unsqueeze(1), enc_out
        ).squeeze(1)

        return context, alpha


class DualAttention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()

        self.history_attn = BahdanauAttention(
            hid_dim, hid_dim, hid_dim
        )
        self.document_attn = BahdanauAttention(
            hid_dim, hid_dim, hid_dim
        )

        self.gate = nn.Sequential(
            nn.Linear(3 * hid_dim, hid_dim),
            nn.Tanh(),
            nn.Linear(hid_dim, hid_dim),
            nn.Sigmoid()
        )

    def forward(
        self, history_out, document_out, hidden,
        history_mask, document_mask
    ):
        h_ctx, h_alpha = self.history_attn(
            history_out, hidden, history_mask
        )
        d_ctx, d_alpha = self.document_attn(
            document_out, hidden, document_mask
        )

        gate = self.gate(
            torch.cat([hidden, h_ctx, d_ctx], dim=-1)
        )

        ctx = gate * d_ctx + (1 - gate) * h_ctx

        return ctx, h_alpha, d_alpha, gate


class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            input_dim,
            emb_dim,
            padding_idx=src_vocab.stoi[PAD]
        )
        self.dropout = nn.Dropout(DROPOUT)
        self.rnn = nn.LSTM(
            emb_dim,
            hid_dim,
            batch_first=True
        )

    def forward(self, x):
        e = self.dropout(self.embedding(x))
        return self.rnn(e)

In [ ]:
# ============================================================
# 9. DECODERS + GROUNDED / UNGROUNDED MODELS
# ============================================================

class UngroundedDecoder(nn.Module):
    def __init__(self, emb_dim, hid_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            len(trg_vocab), emb_dim,
            padding_idx=trg_vocab.stoi[PAD]
        )
        self.attn = BahdanauAttention(
            hid_dim, hid_dim, hid_dim
        )
        self.rnn = nn.LSTM(
            emb_dim + hid_dim,
            hid_dim,
            batch_first=True
        )
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(
            emb_dim + 2 * hid_dim,
            len(trg_vocab)
        )

    def step(self, token, hidden, cell, enc_out, mask):
        emb = self.dropout(
            self.embedding(token)
        ).unsqueeze(1)

        ctx, alpha = self.attn(
            enc_out, hidden, mask
        )

        out, (hidden, cell) = self.rnn(
            torch.cat([emb, ctx.unsqueeze(1)], -1),
            (hidden.unsqueeze(0), cell.unsqueeze(0))
        )

        out = out.squeeze(1)
        hidden, cell = hidden.squeeze(0), cell.squeeze(0)

        logits = self.fc(
            torch.cat(
                [out, ctx, emb.squeeze(1)], -1
            )
        )

        return logits, hidden, cell, alpha


class GroundedDecoder(nn.Module):
    def __init__(self, emb_dim, hid_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            len(trg_vocab), emb_dim,
            padding_idx=trg_vocab.stoi[PAD]
        )
        self.attn = DualAttention(hid_dim)
        self.rnn = nn.LSTM(
            emb_dim + hid_dim,
            hid_dim,
            batch_first=True
        )
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(
            emb_dim + 2 * hid_dim,
            len(trg_vocab)
        )

    def step(
        self, token, hidden, cell,
        history_out, document_out,
        history_mask, document_mask
    ):
        emb = self.dropout(
            self.embedding(token)
        ).unsqueeze(1)

        ctx, h_alpha, d_alpha, gate = self.attn(
            history_out,
            document_out,
            hidden,
            history_mask,
            document_mask
        )

        out, (hidden, cell) = self.rnn(
            torch.cat([emb, ctx.unsqueeze(1)], -1),
            (hidden.unsqueeze(0), cell.unsqueeze(0))
        )

        out = out.squeeze(1)
        hidden, cell = hidden.squeeze(0), cell.squeeze(0)

        logits = self.fc(
            torch.cat(
                [out, ctx, emb.squeeze(1)], -1
            )
        )

        return logits, hidden, cell, h_alpha, d_alpha, gate


class UngroundedSeq2Seq(nn.Module):
    def __init__(self, emb_dim, hid_dim):
        super().__init__()
        self.encoder = Encoder(
            len(src_vocab), emb_dim, hid_dim
        )
        self.decoder = UngroundedDecoder(
            emb_dim, hid_dim
        )

    def forward(self, history, target, teacher_forcing=0.5):
        B, T = target.shape
        outputs = torch.zeros(
            B, T, len(trg_vocab), device=history.device
        )

        enc_out, hidden, cell = self.encoder(history)
        mask = history != src_vocab.stoi[PAD]
        token = target[:, 0]

        for t in range(1, T):
            logits, hidden, cell, _ = self.decoder.step(
                token, hidden, cell, enc_out, mask
            )
            outputs[:, t] = logits
            token = (
                target[:, t]
                if random.random() < teacher_forcing
                else logits.argmax(-1)
            )

        return outputs


class GroundedSeq2Seq(nn.Module):
    def __init__(self, emb_dim, hid_dim):
        super().__init__()

        self.history_encoder = Encoder(
            len(src_vocab), emb_dim, hid_dim
        )
        self.document_encoder = Encoder(
            len(src_vocab), emb_dim, hid_dim
        )
        self.decoder = GroundedDecoder(
            emb_dim, hid_dim
        )

    def forward(
        self, history, document, target,
        teacher_forcing=0.5
    ):
        B, T = target.shape
        outputs = torch.zeros(
            B, T, len(trg_vocab), device=history.device
        )

        h_out, hidden, cell = self.history_encoder(history)
        d_out, _, _ = self.document_encoder(document)

        h_mask = history != src_vocab.stoi[PAD]
        d_mask = document != src_vocab.stoi[PAD]

        token = target[:, 0]

        for t in range(1, T):
            logits, hidden, cell, _, _, _ = self.decoder.step(
                token, hidden, cell,
                h_out, d_out,
                h_mask, d_mask
            )

            outputs[:, t] = logits

            token = (
                target[:, t]
                if random.random() < teacher_forcing
                else logits.argmax(-1)
            )

        return outputs

In [ ]:
# ============================================================
# 10. TRAINING / VALIDATION UTILITIES
# ============================================================

criterion = nn.CrossEntropyLoss(
    ignore_index=trg_vocab.stoi[PAD]
)

def build_model(kind, emb_dim, hid_dim):
    if kind == "grounded":
        return GroundedSeq2Seq(
            emb_dim, hid_dim
        ).to(DEVICE)

    return UngroundedSeq2Seq(
        emb_dim, hid_dim
    ).to(DEVICE)


def run_epoch(model, data_loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total = 0.0

    for batch in data_loader:
        history = batch["history"].to(DEVICE)
        document = batch["document"].to(DEVICE)
        target = batch["target"].to(DEVICE)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            if isinstance(model, GroundedSeq2Seq):
                logits = model(
                    history, document, target,
                    TEACHER_FORCING if training else 0.0
                )
            else:
                logits = model(
                    history, target,
                    TEACHER_FORCING if training else 0.0
                )

            loss = criterion(
                logits[:, 1:].reshape(-1, logits.size(-1)),
                target[:, 1:].reshape(-1)
            )

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), CLIP
                )
                optimizer.step()

        total += loss.item()

    return total / max(1, len(data_loader))


def fit(model, train_loader, val_loader, epochs):
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LR
    )

    best_loss = float("inf")
    best_state = None
    wait = 0
    history = []

    for epoch in range(1, epochs + 1):
        tr_loss = run_epoch(
            model, train_loader, optimizer
        )
        va_loss = run_epoch(
            model, val_loader, None
        )

        history.append({
            "epoch": epoch,
            "train_loss": tr_loss,
            "valid_loss": va_loss
        })

        print(
            f"Epoch {epoch:02d} | "
            f"train={tr_loss:.4f} | valid={va_loss:.4f}"
        )

        if va_loss < best_loss:
            best_loss = va_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            wait = 0
        else:
            wait += 1

        if wait >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return pd.DataFrame(history)

# Stratified grouped CV and OOF

The fold unit is the **conversation**, not the individual turn. Stratification uses a three-level code-mixing intensity bin.

This avoids the common dialogue leakage error:

```text
same conversation → train fold
same conversation → validation fold
```

The OOF table stores predictions only from the fold in which that conversation was held out.

In [ ]:
# ============================================================
# 11. STRATIFIED GROUPED FOLDS
# ============================================================

fold_splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

train_ex = train_ex.reset_index(drop=True)
train_ex["fold"] = -1

for fold, (_, valid_idx) in enumerate(
    fold_splitter.split(
        train_ex,
        y=train_ex["mix_bin"],
        groups=train_ex["conversation_id"]
    )
):
    train_ex.loc[valid_idx, "fold"] = fold

display(
    train_ex.groupby(["fold", "mix_bin"]).size()
)

In [ ]:
# ============================================================
# 12. GREEDY DECODING + OOF TRAINING
# ============================================================

@torch.no_grad()
def greedy_predict(model, data_loader):
    model.eval()
    records = []

    for batch in data_loader:
        history = batch["history"].to(DEVICE)
        document = batch["document"].to(DEVICE)
        B = history.size(0)

        sequences = [[] for _ in range(B)]
        finished = torch.zeros(
            B, dtype=torch.bool, device=DEVICE
        )

        if isinstance(model, GroundedSeq2Seq):
            h_out, hidden, cell = model.history_encoder(history)
            d_out, _, _ = model.document_encoder(document)

            h_mask = history != src_vocab.stoi[PAD]
            d_mask = document != src_vocab.stoi[PAD]

            token = torch.full(
                (B,), trg_vocab.stoi[SOS],
                dtype=torch.long, device=DEVICE
            )

            for _ in range(MAX_TARGET_TOKENS - 1):
                logits, hidden, cell, _, _, _ = model.decoder.step(
                    token, hidden, cell,
                    h_out, d_out, h_mask, d_mask
                )

                nxt = logits.argmax(-1)

                for i, value in enumerate(nxt.tolist()):
                    if not finished[i]:
                        sequences[i].append(value)
                        finished[i] |= value == trg_vocab.stoi[EOS]

                token = nxt
                if finished.all():
                    break

        else:
            enc_out, hidden, cell = model.encoder(history)
            mask = history != src_vocab.stoi[PAD]

            token = torch.full(
                (B,), trg_vocab.stoi[SOS],
                dtype=torch.long, device=DEVICE
            )

            for _ in range(MAX_TARGET_TOKENS - 1):
                logits, hidden, cell, _ = model.decoder.step(
                    token, hidden, cell, enc_out, mask
                )

                nxt = logits.argmax(-1)

                for i, value in enumerate(nxt.tolist()):
                    if not finished[i]:
                        sequences[i].append(value)
                        finished[i] |= value == trg_vocab.stoi[EOS]

                token = nxt
                if finished.all():
                    break

        for i in range(B):
            records.append({
                "row_id": batch["row_id"][i],
                "conversation_id": batch["conversation_id"][i],
                "prediction": trg_vocab.decode(sequences[i])
            })

    return pd.DataFrame(records)


def run_oof(kind, emb_dim=DEFAULT_EMB_DIM, hid_dim=DEFAULT_HID_DIM):
    predictions = []

    for fold in range(N_SPLITS):
        tr = train_ex[train_ex["fold"] != fold]
        va = train_ex[train_ex["fold"] == fold]

        print(f"\n{kind.upper()} | fold {fold + 1}/{N_SPLITS}")

        model = build_model(kind, emb_dim, hid_dim)

        hist = fit(
            model,
            loader(tr, True),
            loader(va, False),
            CV_EPOCHS
        )

        pred = greedy_predict(
            model,
            loader(va, False)
        )

        pred = va[
            [
                "row_id", "conversation_id", "target",
                "hinglish_ratio", "english_ratio"
            ]
        ].merge(
            pred,
            on=["row_id", "conversation_id"],
            how="left"
        )

        pred["fold"] = fold
        predictions.append(pred)

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.concat(predictions, ignore_index=True)


if RUN_CV:
    oof_ungrounded = run_oof("ungrounded")
    oof_grounded = run_oof("grounded")

    print(
        len(oof_ungrounded),
        len(oof_grounded),
        "OOF rows"
    )

In [ ]:
# ============================================================
# 13. OPTIONAL GRID SEARCH
# ============================================================

SEARCH_GRID = [
    {"emb_dim": 128, "hid_dim": 256},
    {"emb_dim": 256, "hid_dim": 384}
]

def grid_search_grounded():
    results = []

    for params in ParameterGrid(SEARCH_GRID):
        losses = []

        for fold in range(N_SPLITS):
            tr = train_ex[train_ex["fold"] != fold]
            va = train_ex[train_ex["fold"] == fold]

            model = build_model(
                "grounded",
                params["emb_dim"],
                params["hid_dim"]
            )

            hist = fit(
                model,
                loader(tr, True),
                loader(va, False),
                max(2, CV_EPOCHS // 2)
            )

            losses.append(hist["valid_loss"].min())

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        results.append({
            **params,
            "mean_valid_loss": float(np.mean(losses)),
            "std_valid_loss": float(np.std(losses))
        })

    return pd.DataFrame(results).sort_values(
        "mean_valid_loss"
    ).reset_index(drop=True)


if RUN_GRID_SEARCH and RUN_CV:
    grid_results = grid_search_grounded()
    display(grid_results)

    BEST_EMB_DIM = int(grid_results.iloc[0]["emb_dim"])
    BEST_HID_DIM = int(grid_results.iloc[0]["hid_dim"])
else:
    BEST_EMB_DIM = DEFAULT_EMB_DIM
    BEST_HID_DIM = DEFAULT_HID_DIM

print(BEST_EMB_DIM, BEST_HID_DIM)

In [ ]:
# ============================================================
# 14. OOF METRICS
# ============================================================

def bleu(df):
    refs = [[tokenize(x)] for x in df["target"]]
    hyps = [tokenize(x) for x in df["prediction"]]
    return corpus_bleu(refs, hyps)

def rouge(df):
    if rouge_scorer is None:
        return {"ROUGE-1": np.nan, "ROUGE-2": np.nan, "ROUGE-L": np.nan}

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=False
    )

    vals = defaultdict(list)

    for ref, pred in zip(df["target"], df["prediction"]):
        score = scorer.score(ref, pred)
        vals["ROUGE-1"].append(score["rouge1"].fmeasure)
        vals["ROUGE-2"].append(score["rouge2"].fmeasure)
        vals["ROUGE-L"].append(score["rougeL"].fmeasure)

    return {k: float(np.mean(v)) for k, v in vals.items()}

if RUN_CV:
    print("Ungrounded OOF:", bleu(oof_ungrounded), rouge(oof_ungrounded))
    print("Grounded OOF:", bleu(oof_grounded), rouge(oof_grounded))

In [ ]:
# ============================================================
# 15. FINAL TRAINING
# ============================================================

ungrounded_model = build_model(
    "ungrounded", BEST_EMB_DIM, BEST_HID_DIM
)

grounded_model = build_model(
    "grounded", BEST_EMB_DIM, BEST_HID_DIM
)

hist_ungrounded = fit(
    ungrounded_model,
    loader(train_ex, True),
    loader(val_ex, False),
    FINAL_EPOCHS
)

hist_grounded = fit(
    grounded_model,
    loader(train_ex, True),
    loader(val_ex, False),
    FINAL_EPOCHS
)

plt.figure(figsize=(8, 5))
plt.plot(
    hist_ungrounded["epoch"],
    hist_ungrounded["valid_loss"],
    label="Ungrounded"
)
plt.plot(
    hist_grounded["epoch"],
    hist_grounded["valid_loss"],
    label="Grounded"
)
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Final model convergence")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
# ============================================================
# 16. TF-IDF RETRIEVAL BASELINE
# ============================================================

def retrieval_text(df):
    return (
        df["history"].fillna("")
        + " [DOCUMENT] "
        + df["document"].fillna("")
    )

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=60000,
    sublinear_tf=True
)

X_train = tfidf.fit_transform(
    retrieval_text(train_ex)
)

def retrieval_predict(query_df):
    Xq = tfidf.transform(retrieval_text(query_df))
    sim = cosine_similarity(Xq, X_train)
    best = sim.argmax(axis=1)

    result = query_df[
        [
            "row_id", "conversation_id", "target",
            "hinglish_ratio", "english_ratio"
        ]
    ].copy()

    result["prediction"] = [
        train_ex.iloc[i]["target"] for i in best
    ]

    result["similarity"] = [
        sim[r, i] for r, i in enumerate(best)
    ]

    return result

retrieval_test = retrieval_predict(test_ex)
print("Retrieval BLEU:", bleu(retrieval_test))

# Beam search

Greedy decoding chooses the locally highest-probability next token. Beam search keeps several partial responses:

\[
S(y_{1:t})=\sum_{k=1}^{t}\log P(y_k|y_{<k},x)
\]

and uses length-normalized scores at the end. The final comparison therefore includes both greedy and Beam-5 decoding.

In [ ]:
# ============================================================
# 17. BEAM SEARCH FOR GROUNDED MODEL
# ============================================================

@torch.no_grad()
def beam_one(model, history_ids, document_ids):
    history = torch.tensor(
        [history_ids], dtype=torch.long, device=DEVICE
    )
    document = torch.tensor(
        [document_ids], dtype=torch.long, device=DEVICE
    )

    h_out, hidden, cell = model.history_encoder(history)
    d_out, _, _ = model.document_encoder(document)

    h_mask = history != src_vocab.stoi[PAD]
    d_mask = document != src_vocab.stoi[PAD]

    beams = [(
        [trg_vocab.stoi[SOS]],
        0.0,
        hidden,
        cell
    )]

    for _ in range(MAX_TARGET_TOKENS - 1):
        candidates = []

        for seq, score, h, c in beams:
            if seq[-1] == trg_vocab.stoi[EOS]:
                candidates.append((seq, score, h, c))
                continue

            token = torch.tensor(
                [seq[-1]], dtype=torch.long, device=DEVICE
            )

            logits, nh, nc, _, _, _ = model.decoder.step(
                token, h[0], c[0],
                h_out, d_out, h_mask, d_mask
            )

            logp = F.log_softmax(logits, dim=-1)

            values, indices = torch.topk(
                logp, BEAM_WIDTH, dim=-1
            )

            for value, index in zip(values[0], indices[0]):
                candidates.append((
                    seq + [int(index)],
                    score + float(value),
                    nh.unsqueeze(0),
                    nc.unsqueeze(0)
                ))

        candidates.sort(
            key=lambda x:
                x[1] / (len(x[0]) ** LENGTH_PENALTY),
            reverse=True
        )

        beams = candidates[:BEAM_WIDTH]

        if all(
            x[0][-1] == trg_vocab.stoi[EOS]
            for x in beams
        ):
            break

    best = max(
        beams,
        key=lambda x:
            x[1] / (len(x[0]) ** LENGTH_PENALTY)
    )

    return trg_vocab.decode(best[0][1:])


beam_rows = []

for _, row in test_ex.iterrows():
    pred = beam_one(
        grounded_model,
        src_vocab.encode(
            row["history"], MAX_HISTORY_TOKENS
        ),
        src_vocab.encode(
            row["document"], MAX_DOCUMENT_TOKENS
        )
    )

    beam_rows.append({
        "row_id": row["row_id"],
        "conversation_id": row["conversation_id"],
        "target": row["target"],
        "prediction": pred,
        "history": row["history"],
        "document": row["document"],
        "hinglish_ratio": row["hinglish_ratio"],
        "english_ratio": row["english_ratio"]
    })

test_beam = pd.DataFrame(beam_rows)

print("Beam-5 BLEU:", bleu(test_beam))

In [ ]:
# ============================================================
# 18. FINAL TEST COMPARISON
# ============================================================

test_loader = loader(test_ex, False)

test_ungrounded = greedy_predict(
    ungrounded_model, test_loader
).merge(
    test_ex[
        [
            "row_id", "conversation_id", "target",
            "history", "document",
            "hinglish_ratio", "english_ratio"
        ]
    ],
    on=["row_id", "conversation_id"],
    how="left"
)

test_grounded = greedy_predict(
    grounded_model, test_loader
).merge(
    test_ex[
        [
            "row_id", "conversation_id", "target",
            "history", "document",
            "hinglish_ratio", "english_ratio"
        ]
    ],
    on=["row_id", "conversation_id"],
    how="left"
)

def metric_row(name, df):
    return {
        "system": name,
        "BLEU": bleu(df),
        **rouge(df)
    }

test_metrics = pd.DataFrame([
    metric_row("TF-IDF Retrieval", retrieval_test),
    metric_row("Ungrounded LSTM + Bahdanau", test_ungrounded),
    metric_row(
        "Grounded Dual-Source LSTM + Bahdanau — Greedy",
        test_grounded
    ),
    metric_row(
        "Grounded Dual-Source LSTM + Bahdanau — Beam-5",
        test_beam
    )
])

display(test_metrics)

In [ ]:
# ============================================================
# 19. GROUNDING DIAGNOSTICS
# ============================================================

STOPWORDS = set("""
a an the is are was were be been i you he she it we they and or
but if this that those these in on at to for of with from by as
into about very really just do does did have has had
""".split())

def content_tokens(text):
    return {
        t for t in tokenize(text)
        if t.isalnum() and len(t) > 2 and t not in STOPWORDS
    }

def overlap(pred, source):
    p, s = content_tokens(pred), content_tokens(source)
    return len(p & s) / max(1, len(p))

for df in [test_grounded, test_beam]:
    df["document_overlap"] = [
        overlap(p, d)
        for p, d in zip(df["prediction"], df["document"])
    ]
    df["history_overlap"] = [
        overlap(p, h)
        for p, h in zip(df["prediction"], df["history"])
    ]

display(pd.DataFrame({
    "system": ["Grounded Greedy", "Grounded Beam-5"],
    "mean_document_overlap": [
        test_grounded["document_overlap"].mean(),
        test_beam["document_overlap"].mean()
    ],
    "mean_history_overlap": [
        test_grounded["history_overlap"].mean(),
        test_beam["history_overlap"].mean()
    ]
}))

In [ ]:
# ============================================================
# 20. CODE-MIXING DEGRADATION ANALYSIS
# ============================================================

tmp = test_grounded.copy()

tmp["mix_bucket"] = pd.qcut(
    tmp["hinglish_ratio"].rank(method="first"),
    q=3,
    labels=[
        "Low Hinglish",
        "Medium Hinglish",
        "High Hinglish"
    ]
)

rows = []

for bucket, group in tmp.groupby(
    "mix_bucket", observed=False
):
    rows.append({
        "bucket": str(bucket),
        "n": len(group),
        "BLEU": bleu(group),
        "target_tokens": np.mean([
            len(tokenize(x)) for x in group["target"]
        ]),
        "prediction_tokens": np.mean([
            len(tokenize(x)) for x in group["prediction"]
        ])
    })

mix_table = pd.DataFrame(rows)
display(mix_table)

plt.figure(figsize=(8, 5))
plt.bar(mix_table["bucket"], mix_table["BLEU"])
plt.xlabel("Code-mixing intensity")
plt.ylabel("BLEU")
plt.title("Grounded model performance vs Hinglish intensity")
plt.grid(axis="y", alpha=0.25)
plt.show()

In [ ]:
# ============================================================
# 21. QUALITATIVE GROUNDED VS UNGROUNDED
# ============================================================

qualitative = test_grounded[
    [
        "row_id",
        "history",
        "target",
        "prediction",
        "document",
        "hinglish_ratio"
    ]
].merge(
    test_ungrounded[
        ["row_id", "prediction"]
    ].rename(
        columns={"prediction": "ungrounded_prediction"}
    ),
    on="row_id",
    how="left"
).sort_values(
    "hinglish_ratio", ascending=False
)

display(
    qualitative[
        [
            "history",
            "target",
            "ungrounded_prediction",
            "prediction",
            "hinglish_ratio"
        ]
    ].head(20)
)

# Interpretation checklist

When writing the report, inspect whether the grounded model:

- uses entities present in the WikiData;
- answers document-dependent questions better than the ungrounded model;
- remains conversational rather than copying document prose;
- becomes weaker on highly code-mixed turns;
- produces generic replies despite having useful document context;
- hallucinates facts absent from the document.

BLEU/ROUGE should not be treated as the sole definition of dialogue quality because many different replies can be valid.

In [ ]:
# ============================================================
# 22. SAVE ARTIFACTS
# ============================================================

if RUN_CV:
    oof_ungrounded.to_csv(
        ARTIFACT_DIR / "ungrounded_oof_predictions.csv",
        index=False
    )
    oof_grounded.to_csv(
        ARTIFACT_DIR / "grounded_oof_predictions.csv",
        index=False
    )

retrieval_test.to_csv(
    ARTIFACT_DIR / "retrieval_test_predictions.csv",
    index=False
)

test_ungrounded.to_csv(
    ARTIFACT_DIR / "ungrounded_test_predictions.csv",
    index=False
)

test_grounded.to_csv(
    ARTIFACT_DIR / "grounded_greedy_predictions.csv",
    index=False
)

test_beam.to_csv(
    ARTIFACT_DIR / "grounded_beam_predictions.csv",
    index=False
)

test_metrics.to_csv(
    ARTIFACT_DIR / "test_metrics.csv",
    index=False
)

torch.save(
    grounded_model.state_dict(),
    ARTIFACT_DIR / "grounded_dual_source_lstm.pt"
)

torch.save(
    ungrounded_model.state_dict(),
    ARTIFACT_DIR / "ungrounded_lstm.pt"
)

print("Saved:", ARTIFACT_DIR)

# Final design summary

### Baseline 1 — TF-IDF retrieval
Retrieves an observed response using similarity between dialogue + document context.

### Baseline 2 — ungrounded generator
Uses the Experiment-2 attention architecture but only sees dialogue history.

### Full model — document-grounded generator
Uses separate history/document LSTM encoders, two Bahdanau attentions, and a learned source gate.

### Validation protocol
`StratifiedGroupKFold` stratifies code-mixing intensity while grouping complete conversations. OOF predictions are generated from held-out conversations only.

### Model selection
A small `ParameterGrid` compares learned embedding dimensions and hidden sizes. The official validation split remains outside CV and the final test split is used only for final evaluation.

### Evaluation
BLEU and ROUGE are accompanied by document-overlap, history-overlap, code-mixing bucket analysis, and qualitative grounded-vs-ungrounded comparisons.

This keeps the experiment a genuine extension of Experiment 2 while giving it the additional complexity required by multi-turn, document-grounded, code-mixed dialogue.